In [ ]:
"""
ELECTRE TRI-rC: Complete calculations for all countries.

Outputs:
1. tri_rc_assignments.csv
2. tri_rc_profile_comparisons.csv
3. tri_rc_criterion_calculations.csv

IMPORTANT:
The five expert profiles are named b1,...,b5.
The lecture's lower-bound assignment rule also refers to b0.
Because no separate b0 was supplied, b0 is constructed from the dataset
as the worst observed value on each criterion (minimum for benefits,
maximum for costs). This is an explicit, transparent modelling assumption.

The adopted veto rule is binary:
- If any applicable veto is triggered, D = 0.
- Otherwise, D = 1.

A positive oriented difference always means the first object
performs better than the second.
"""

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. INPUT AND OUTPUT FILES
# ============================================================



BASE_DIR = Path.cwd().parent

INPUT_FILE = BASE_DIR / "Initial data" / "LPI_final.csv"

OUTPUT_ASSIGNMENTS = BASE_DIR / "7. electre_tri_rc_fixed_outputs" /"new_tri_rc_assignments.csv"
OUTPUT_COMPARISONS = BASE_DIR / "7. electre_tri_rc_fixed_outputs" / "new_tri_rc_profile_comparisons.csv"
OUTPUT_CRITERIA = BASE_DIR / "7. electre_tri_rc_fixed_outputs" / "new_tri_rc_criterion_calculations.csv"


# ============================================================
# 2. CRITERIA
# ============================================================

CRITERIA = ["C1", "C2", "C3", "C4", "C5", "C6"]

COLUMN_MAP = {
    "C1": "Maritime connectivity",
    "C2": "Container dwell time",
    "C3": "Aviation connectivity",
    "C4": "Aviation dwell time",
    "C5": "B2B postal connectivity",
    "C6": "B2B postal delivery time",
}

# Higher values are better for connectivity indicators.
BENEFIT = {"C1", "C3", "C5"}

# Lower values are better for time indicators.
COST = {"C2", "C4", "C6"}


# ============================================================
# 3. AGGREGATED BWM WEIGHTS
# ============================================================

WEIGHTS = pd.Series(
    {
        "C1": 0.1858,
        "C2": 0.2524,
        "C3": 0.1358,
        "C4": 0.2166,
        "C5": 0.0629,
        "C6": 0.1462,
    },
    dtype=float,
)

# Normalize the rounded weights so that their sum is exactly 1.
WEIGHTS = WEIGHTS / WEIGHTS.sum()


# ============================================================
# 4. INDifference, PREFERENCE, AND VETO THRESHOLDS
# ============================================================

Q = pd.Series(
    {
        "C1": 3.50,
        "C2": 1.25,
        "C3": 5.50,
        "C4": 1.25,
        "C5": 5.00,
        "C6": 1.15,
    }
)

P = pd.Series(
    {
        "C1": 10.00,
        "C2": 4.00,
        "C3": 14.00,
        "C4": 2.50,
        "C5": 16.00,
        "C6": 3.00,
    }
)

# None means that no veto threshold is applied to that criterion.
V = {
    "C1": None,
    "C2": 6.00,
    "C3": None,
    "C4": 5.50,
    "C5": 25.00,
    "C6": 6.00,
}

LAMBDA = 0.70


# ============================================================
# 5. AGGREGATED CHARACTERISTIC PROFILES
# ============================================================

PROFILE_VALUES = {
    "Class 1": {
        "C1": 24.0,
        "C2": 14.0,
        "C3": 60.5,
        "C4": 5.25,
        "C5": 37.5,
        "C6": 14.0,
    },

    "Class 2": {
        "C1": 36.5,
        "C2": 12.0,
        "C3": 80.0,
        "C4": 4.00,
        "C5": 56.0,
        "C6": 8.0,
    },

    "Class 3": {
        "C1": 51.0,
        "C2": 8.5,
        "C3": 102.0,
        "C4": 3.00,
        "C5": 79.5,
        "C6": 5.5,
    },

    "Class 4": {
        "C1": 66.0,
        "C2": 5.0,
        "C3": 120.5,
        "C4": 2.25,
        "C5": 107.5,
        "C6": 4.0,
    },

    "Class 5": {
        "C1": 71.5,
        "C2": 2.5,
        "C3": 128.5,
        "C4": 1.45,
        "C5": 115.5,
        "C6": 2.0,
    },
}

PROFILE_ORDER = list(PROFILE_VALUES.keys())

# Map the five class profiles to b1, b2, ..., b5.
PROFILES = {
    f"b{i}": PROFILE_VALUES[name]
    for i, name in enumerate(PROFILE_ORDER, start=1)
}

# Boundary assumption:
# b0 duplicates b1 because a separate b0 was not supplied.
PROFILES["b0"] = dict(PROFILES["b1"])


# ============================================================
# 6. ORIENTED PERFORMANCE DIFFERENCE
# ============================================================

def oriented_difference(first, second, criterion):
    """
    Calculate the oriented performance difference.

    Positive difference = first object performs better.
    Negative difference = first object performs worse.

    Benefit criterion:
        first value - second value

    Cost criterion:
        second value - first value
    """

    if criterion in BENEFIT:
        return float(first - second)

    return float(second - first)


# ============================================================
# 7. LOCAL CONCORDANCE
# ============================================================

def local_concordance(delta, criterion):
    """
    Calculate local concordance using q and p.

    If delta >= -q:
        concordance = 1

    If delta <= -p:
        concordance = 0

    Otherwise:
        concordance = (delta + p) / (p - q)
    """

    q = float(Q[criterion])
    p = float(P[criterion])

    if delta >= -q:
        return 1.0

    if delta <= -p:
        return 0.0

    return (delta + p) / (p - q)


# ============================================================
# 8. CALCULATE ONE OUTRANKING DIRECTION
# ============================================================

def calculate_direction(
    first_values,
    second_values,
    first_name,
    second_name,
):
    """
    Calculate all ELECTRE indices for:

        first object S second object

    Returns:
        - Comprehensive concordance
        - Veto criteria
        - Non-discordance
        - Credibility
        - Outranking decision
        - Criterion-level calculations
    """

    details = []

    concordance_sum = 0.0
    vetoes = []

    for criterion in CRITERIA:

        # Step 1: Oriented difference
        delta = oriented_difference(
            first_values[criterion],
            second_values[criterion],
            criterion,
        )

        # Step 2: Local concordance
        concordance = local_concordance(
            delta,
            criterion,
        )

        # Step 3: Weighted contribution
        weight = float(WEIGHTS[criterion])

        contribution = weight * concordance

        concordance_sum += contribution

        # Step 4: Check veto
        veto = False

        if V[criterion] is not None:

            veto_threshold = float(V[criterion])

            if delta <= -veto_threshold:
                veto = True
                vetoes.append(criterion)

        # Save the calculation for this criterion.
        details.append(
            {
                "first_object": first_name,
                "second_object": second_name,
                "criterion": criterion,
                "first_raw_value": float(
                    first_values[criterion]
                ),
                "second_raw_value": float(
                    second_values[criterion]
                ),
                "oriented_difference": delta,
                "q": float(Q[criterion]),
                "p": float(P[criterion]),
                "v": V[criterion],
                "local_concordance": concordance,
                "weight": weight,
                "weighted_contribution": contribution,
                "veto_triggered_on_criterion": veto,
            }
        )

    # Step 5: Non-discordance
    if vetoes:
        non_discordance = 0.0
    else:
        non_discordance = 1.0

    # Step 6: Credibility
    credibility = concordance_sum * non_discordance

    # Step 7: Apply cutting level
    outranks = credibility >= LAMBDA

    summary = {
        "first_object": first_name,
        "second_object": second_name,
        "comprehensive_concordance": concordance_sum,
        "veto_criteria": (
            ",".join(vetoes) if vetoes else "None"
        ),
        "non_discordance_D": non_discordance,
        "credibility": credibility,
        "lambda": LAMBDA,
        "first_outranks_second": bool(outranks),
    }

    return summary, details


# ============================================================
# 9. STRICT PREFERENCE
# ============================================================

def strict_preference(sigma_ab, sigma_ba):
    """
    a is strictly preferred to b when:
        a outranks b
        AND
        b does not outrank a
    """

    return (
        sigma_ab >= LAMBDA
        and not (sigma_ba >= LAMBDA)
    )


# ============================================================
# 10. FORMAL TRI-rC CLASS ASSIGNMENT
# ============================================================

def assign_tri_rc(profile_sigmas):
    """
    Apply the lecture's lower- and upper-class rules.

    profile_sigmas maps each profile to:
        (credibility alternative -> profile,
         credibility profile -> alternative)

    Lower bound:
        Search h = 4, 3, 2, 1, 0.

        Conditions:
        1. Alternative strictly outranks b_h.
        2. sigma(a, b_(h+1)) > sigma(b_h, a).

        If satisfied:
            lower class = h + 1

    Upper bound:
        Search h = 2, 3, 4, 5.

        Conditions:
        1. Profile b_h strictly outranks alternative.
        2. sigma(b_(h-1), a) > sigma(a, b_h).

        If satisfied:
            upper class = h - 1

    IMPORTANT:
    The code uses explicit boundary fallbacks if no profile
    satisfies a rule. Those fallbacks are modelling conventions
    and should be reviewed against your instructor's requirements.
    """

    # --------------------------------------------------------
    # Lower class
    # --------------------------------------------------------

    lower = None
    lower_trace = []

    for h in range(4, -1, -1):

        # Credibility of alternative against current profile.
        sigma_a_bh, sigma_bh_a = profile_sigmas[f"b{h}"]

        # Credibility of alternative against next profile.
        sigma_a_bh_next, _ = profile_sigmas[f"b{h + 1}"]

        condition_1 = strict_preference(
            sigma_a_bh,
            sigma_bh_a,
        )

        condition_2 = (
            sigma_a_bh_next > sigma_bh_a
        )

        lower_trace.append(
            {
                "h": h,
                "profile_checked": f"b{h}",
                "strict_alternative_preference": condition_1,
                "sigma_alt_next_profile": sigma_a_bh_next,
                "sigma_current_profile_alt": sigma_bh_a,
                "second_condition": condition_2,
                "both_conditions_met": (
                    condition_1 and condition_2
                ),
            }
        )

        if condition_1 and condition_2:
            lower = h + 1
            break

    # Explicit fallback if no lower-bound condition is met.
    if lower is None:
        lower = 1

    # --------------------------------------------------------
    # Upper class
    # --------------------------------------------------------

    upper = None
    upper_trace = []

    for h in range(2, 6):

        # Credibility of alternative against current profile.
        sigma_a_bh, sigma_bh_a = profile_sigmas[f"b{h}"]

        # Credibility of previous profile against alternative.
        _, sigma_b_previous_a = profile_sigmas[f"b{h - 1}"]

        condition_1 = strict_preference(
            sigma_bh_a,
            sigma_a_bh,
        )

        condition_2 = (
            sigma_b_previous_a > sigma_a_bh
        )

        upper_trace.append(
            {
                "h": h,
                "profile_checked": f"b{h}",
                "strict_profile_preference": condition_1,
                "sigma_previous_profile_alt": sigma_b_previous_a,
                "sigma_alt_current_profile": sigma_a_bh,
                "second_condition": condition_2,
                "both_conditions_met": (
                    condition_1 and condition_2
                ),
            }
        )

        if condition_1 and condition_2:
            upper = h - 1
            break

    # Explicit fallback if no upper-bound condition is met.
    if upper is None:
        upper = 5

    # Do not silently swap inconsistent bounds.
    bounds_consistent = lower <= upper

    return (
        lower,
        upper,
        lower_trace,
        upper_trace,
        bounds_consistent,
    )


# ============================================================
# 11. MAIN CALCULATION FOR ALL COUNTRIES
# ============================================================

def main():

    # --------------------------------------------------------
    # Load and validate dataset
    # --------------------------------------------------------

    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"Input file not found: {INPUT_FILE}\n"
            "Place LPI_2024_cleaned_complete.csv in the same folder "
            "as this Python script."
        )

    df = pd.read_csv(INPUT_FILE)

    # Construct b0 as the worst observed performance across the dataset.
    # This does not claim that experts supplied b0; it is an explicit boundary assumption.
    boundary = {}
    for criterion, column in COLUMN_MAP.items():
        if criterion in BENEFIT:
            boundary[criterion] = float(df[column].min())
        else:
            boundary[criterion] = float(df[column].max())
    PROFILES["b0"] = boundary

    missing_columns = [
        column
        for column in COLUMN_MAP.values()
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns in the dataset: {missing_columns}"
        )

    if df[list(COLUMN_MAP.values())].isna().any().any():
        raise ValueError(
            "The performance matrix contains missing values. "
            "Use the cleaned complete dataset before running."
        )

    comparison_rows = []
    criterion_rows = []
    assignment_rows = []

    # --------------------------------------------------------
    # Loop through every country
    # --------------------------------------------------------

    for _, row in df.iterrows():

        country_code = row.get("REF_AREA", "")

        country = row.get(
            "REF_AREA_LABEL",
            country_code,
        )

        # Read the six performance values.
        alternative = {
            criterion: float(row[column])
            for criterion, column in COLUMN_MAP.items()
        }

        # Store forward/reverse credibility for each profile.
        profile_sigmas = {}

        # Keep detailed comparison summaries.
        direction_summary = {}

        # ----------------------------------------------------
        # Compare country against b0, b1, ..., b5
        # ----------------------------------------------------

        for h in range(0, 6):

            profile_key = f"b{h}"

            profile = PROFILES[profile_key]

            if h == 0:
                profile_label = (
                    "Boundary b0 (worsed observed value)"
                )
            else:
                profile_label = f"Class {h}"

            # Forward:
            # Country compared with profile.
            forward, forward_details = calculate_direction(
                alternative,
                profile,
                country,
                profile_label,
            )

            # Reverse:
            # Profile compared with country.
            reverse, reverse_details = calculate_direction(
                profile,
                alternative,
                profile_label,
                country,
            )

            profile_sigmas[profile_key] = (
                forward["credibility"],
                reverse["credibility"],
            )

            direction_summary[profile_key] = (
                forward,
                reverse,
            )

            # Save both comparison directions.
            for direction, result, details in [
                (
                    "alternative_vs_profile",
                    forward,
                    forward_details,
                ),
                (
                    "profile_vs_alternative",
                    reverse,
                    reverse_details,
                ),
            ]:

                comparison_rows.append(
                    {
                        "REF_AREA": country_code,
                        "country": country,
                        "profile_index": profile_key,
                        "profile_label": profile_label,
                        "direction": direction,
                        **result,
                    }
                )

                # Save criterion-level calculations.
                for detail in details:

                    criterion_rows.append(
                        {
                            "REF_AREA": country_code,
                            "country": country,
                            "profile_index": profile_key,
                            "profile_label": profile_label,
                            "direction": direction,
                            **detail,
                        }
                    )

        # ----------------------------------------------------
        # Calculate formal lower/upper class bounds
        # ----------------------------------------------------

        (
            lower,
            upper,
            lower_trace,
            upper_trace,
            bounds_consistent,
        ) = assign_tri_rc(profile_sigmas)

        assignment = {
            "REF_AREA": country_code,
            "country": country,
            "lower_class": lower,
            "upper_class": upper,
            "class_interval": f"[{lower}, {upper}]",
            "precise_assignment": bool(lower == upper),
            "bounds_consistent": bool(bounds_consistent),
            "lambda": LAMBDA,
            "b0_boundary_assumption": (
                "b0 = worst observed dataset values (min benefits, max costs); "
                "explicit modelling assumption, not expert-elicited"
            ),
        }

        # Add forward/reverse credibility for each expert profile.
        for h in range(1, 6):

            forward, reverse = direction_summary[f"b{h}"]

            assignment[f"sigma_alt_Class{h}"] = (
                forward["credibility"]
            )

            assignment[f"sigma_Class{h}_alt"] = (
                reverse["credibility"]
            )

            assignment[f"veto_alt_Class{h}"] = (
                forward["veto_criteria"]
            )

            assignment[f"veto_Class{h}_alt"] = (
                reverse["veto_criteria"]
            )

        # Store assignment rule traces for auditability.
        assignment["lower_rule_trace"] = str(lower_trace)
        assignment["upper_rule_trace"] = str(upper_trace)

        assignment_rows.append(assignment)

    # ========================================================
    # 12. EXPORT RESULTS
    # ========================================================

    assignments_df = pd.DataFrame(assignment_rows)
    comparisons_df = pd.DataFrame(comparison_rows)
    criteria_df = pd.DataFrame(criterion_rows)

    assignments_df.to_csv(
        OUTPUT_ASSIGNMENTS,
        index=False,
    )

    comparisons_df.to_csv(
        OUTPUT_COMPARISONS,
        index=False,
    )

    criteria_df.to_csv(
        OUTPUT_CRITERIA,
        index=False,
    )

    # ========================================================
    # 13. PRINT SUMMARY
    # ========================================================

    print("=" * 65)
    print("ELECTRE TRI-rC ANALYSIS COMPLETE")
    print("=" * 65)

    print(f"\nNumber of economies processed: {len(df)}")
    print(f"Sum of normalized weights: {WEIGHTS.sum():.6f}")
    print(f"Cutting level lambda: {LAMBDA:.2f}")

    print("\nClass interval counts:")
    print(
        assignments_df["class_interval"]
        .value_counts()
        .sort_index()
        .to_string()
    )

    print("\nInconsistent lower/upper bounds:")

    inconsistent = assignments_df.loc[
        ~assignments_df["bounds_consistent"],
        [
            "REF_AREA",
            "country",
            "class_interval",
        ],
    ]

    if inconsistent.empty:
        print("None")
    else:
        print(inconsistent.to_string(index=False))

    print("\nUSA assignment:")

    usa = assignments_df[
        assignments_df["REF_AREA"]
        .astype(str)
        .str.upper()
        .eq("USA")
    ]

    if usa.empty:
        print("USA not found. Check the REF_AREA codes.")
    else:
        print(
            usa[
                [
                    "country",
                    "lower_class",
                    "upper_class",
                    "class_interval",
                    "precise_assignment",
                    "bounds_consistent",
                ]
            ].to_string(index=False)
        )

    print("\nOutput files saved:")

    print(f"1. {OUTPUT_ASSIGNMENTS}")
    print(f"2. {OUTPUT_COMPARISONS}")
    print(f"3. {OUTPUT_CRITERIA}")

    print("\nImportant modelling assumption:")
    print(
        "b0 is constructed from worst observed dataset values "
        "(minimum for benefits, maximum for costs). This is an explicit modelling assumption."
    )


# ============================================================
# 14. RUN THE PROGRAM
# ============================================================

if __name__ == "__main__":
    main()

ELECTRE TRI-rC ANALYSIS COMPLETE

Number of economies processed: 96
Sum of normalized weights: 1.000000
Cutting level lambda: 0.70

Class interval counts:
class_interval
[1, 1]     7
[1, 2]    15
[1, 3]     5
[1, 4]     2
[2, 2]    12
[2, 3]     2
[2, 4]     1
[3, 3]    23
[4, 4]    10
[4, 5]     3
[5, 5]    16

Inconsistent lower/upper bounds:
None

USA assignment:
      country  lower_class  upper_class class_interval  precise_assignment  bounds_consistent
United States            5            5         [5, 5]                True               True

Output files saved:
1. e:\TU DELFT\Year 1\Q1\MCDA\asgn_2\MCDA-Assignment-2\7. electre_tri_rc_fixed_outputs\new_tri_rc_assignments.csv
2. e:\TU DELFT\Year 1\Q1\MCDA\asgn_2\MCDA-Assignment-2\7. electre_tri_rc_fixed_outputs\new_tri_rc_profile_comparisons.csv
3. e:\TU DELFT\Year 1\Q1\MCDA\asgn_2\MCDA-Assignment-2\7. electre_tri_rc_fixed_outputs\new_tri_rc_criterion_calculations.csv

Important modelling assumption:
b0 is constructed from worst